<a href="https://colab.research.google.com/github/keshav123333/amazon-ml/blob/main/Amazon_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np

In [2]:
url="http://raw.githubusercontent.com/dixitkeshav/AMAZON_ML/refs/heads/main/new/student_resource/dataset/train.csv"
df_train=pd.read_csv(url)

In [3]:
df_train

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.890
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.120
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.970
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.340
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.490
...,...,...,...,...
74995,41424,Item Name: ICE BREAKERS Spearmint Sugar Free M...,https://m.media-amazon.com/images/I/81p9PcPsff...,10.395
74996,35537,"Item Name: Davidson's Organics, Vanilla Essenc...",https://m.media-amazon.com/images/I/51DDKoa+mb...,35.920
74997,249971,Item Name: Jolly Rancher Hard Candy - Blue Ras...,https://m.media-amazon.com/images/I/91R2XCcpUf...,50.330
74998,188322,Item Name: Nescafe Dolce Gusto Capsules - CARA...,https://m.media-amazon.com/images/I/51W40YU98+...,15.275


In [6]:
df_train["catalog_content"][1]

'Item Name: Salerno Cookies, The Original Butter Cookies, 8 Ounce (Pack of 4)\nBullet Point 1: Original Butter Cookies: Classic butter cookies made with real butter\nBullet Point 2: Variety Pack: Includes 4 boxes with 32 cookies total\nBullet Point 3: Occasion Perfect: Delicious cookies for birthdays, weddings, anniversaries\nBullet Point 4: Shareable Treats: Fun to give and enjoy with friends and family\nBullet Point 5: Salerno Brand: Trusted brand of delicious butter cookies since 1925\nValue: 32.0\nUnit: Ounce\n'

In [ ]:
import os
import requests
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm
import pandas as pd # Make sure to have your dataframe loaded, e.g., df_train

# --- KEY CHANGE: Save images to your Google Drive for persistence ---
# This folder will be created inside 'My Drive'
GDRIVE_IMAGE_DIR = '/content/drive/MyDrive/smart_product_pricing_images'
os.makedirs(GDRIVE_IMAGE_DIR, exist_ok=True)

# Let's assume df_train is your training dataframe
# df_train = pd.read_csv('dataset/train.csv')

downloaded_image_paths = {}
error_count = 0

# --- KEY CHANGE: Use tqdm for a progress bar ---
# We iterate over a list of tuples for better performance with tqdm
image_list = list(df_train[['sample_id', 'image_link']].itertuples(index=False, name=None))

for sample_id, image_link in tqdm(image_list, desc="Downloading Images"):
    image_path = os.path.join(GDRIVE_IMAGE_DIR, f'{sample_id}.jpg') # Save as JPG for efficiency
    downloaded_image_paths[sample_id] = image_path # Store the path for later use

    # --- KEY CHANGE: Skip if the image already exists ---
    if os.path.exists(image_path):
        continue # Don't download again

    try:
        response = requests.get(image_link, timeout=10)
        response.raise_for_status()

        if 'image' in response.headers.get('Content-Type', ''):
            image = Image.open(BytesIO(response.content))

            # --- OPTIMIZATION: Resize image to save space and standardize for ML models ---
            image = image.convert('RGB') # Convert to RGB to avoid issues with transparency (PNG)
            resized_image = image.resize((224, 224))
            resized_image.save(image_path, 'JPEG') # Save as JPEG

        else:
            # Silently skip non-image content or handle as needed
            error_count += 1

    except (requests.exceptions.RequestException, IOError):
        # Silently skip download/processing errors
        error_count += 1


print("\n--- Download Summary ---")
print(f"Image directory: {GDRIVE_IMAGE_DIR}")
print(f"Total images in dataframe: {len(df_train)}")
# Count how many files were actually saved in the directory
successful_downloads = len(os.listdir(GDRIVE_IMAGE_DIR))
print(f"Images successfully saved in Drive: {successful_downloads}")
print(f"Errors or non-image links skipped: {error_count}")


--- Download Summary ---
Image directory: /content/drive/MyDrive/smart_product_pricing_images
Total images in dataframe: 5000
Images successfully saved in Drive: 5000
Errors or non-image links skipped: 0


In [4]:
from transformers import AutoTokenizer,AutoModel
import torch
from tqdm import tqdm


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Better model
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2")
model1 = AutoModel.from_pretrained("sentence-transformers/all-mpnet-base-v2").to(device)
model1.eval()

def get_text_embeddings_mpnet(texts):
    embeddings = []
    for text in tqdm(texts):
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            outputs = model1(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
            embeddings.append(emb)
    return embeddings

# text_features = get_text_embeddings_mpnet(df_train['catalog_content'].tolist())


ye dusra

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model1 = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(device)
model1.eval()

def get_text_embeddings_all_Mini(texts):
    embeddings = []
    for text in tqdm(texts):
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model1(**inputs)

            emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
            embeddings.append(emb)
    return embeddings

text_features = get_text_embeddings_all_Mini(df_train['catalog_content'].tolist())

100%|██████████| 75000/75000 [07:04<00:00, 176.49it/s]


effiecent net

if link se direct laani hai toh

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm

effnet = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')

def get_image_embedding_effnet_imageLink(img_url):
    try:
        response = requests.get(img_url, timeout=8)
        if response.status_code != 200:
            raise ValueError("Bad response")
        img = Image.open(BytesIO(response.content)).convert('RGB')
        img = img.resize((224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)  # very important
        feat = effnet.predict(x, verbose=0)
        return feat.flatten()
    except Exception as e:
        print(f"⚠️ Failed: {img_url} | {e}")
        return np.zeros((1280,))

# Example:
image_features = [get_image_embedding_effnet_imageLink(url) for url in tqdm(df_train['image_link'])]
image_features = np.array(image_features)

print("Shape:", image_features.shape)
print("Max value:", image_features.max())
print("Zero vectors:", np.sum(np.all(image_features == 0, axis=1)))


In [ ]:
image_features.shape

(10000, 1280)

for if pehle download kar lo toh

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
import os
from tqdm import tqdm
from PIL import Image

# Path to your image folder
image_folder = '/content/drive/MyDrive/smart_product_pricing_images'

# Load EfficientNetB0 (no top layer, global avg pooling)
effnet = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')

def get_image_embedding_from_path(img_path):
    try:
        img = Image.open(img_path).convert('RGB')
        img = img.resize((224, 224))  # EfficientNetB0 expects 224x224
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        feat = effnet.predict(x, verbose=0)
        return feat.flatten()
    except:
        return np.zeros((1280,))  # EfficientNetB0 outputs 1280-dim features

# Get all image paths
image_paths = [os.path.join(image_folder, f)
               for f in os.listdir(image_folder)
               if f.lower().endswith(('png', 'jpg', 'jpeg'))]

# Extract features
image_features = []
for path in tqdm(image_paths):
    image_features.append(get_image_embedding_from_path(path))

image_features = np.array(image_features)
print("Shape of image features:", image_features.shape)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


100%|██████████| 5000/5000 [06:55<00:00, 12.04it/s]

Shape of image features: (5000, 1280)


In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
import os
from tqdm import tqdm
from PIL import Image

# Path to your folder containing 5000 images
image_folder = '/content/drive/MyDrive/smart_product_pricing_images'

# Load VGG16 model (without top layers, with avg pooling)
vgg = VGG16(weights='imagenet', include_top=False, pooling='avg')  # Output: 512-dim vector

def get_image_embedding_from_path(img_path):
    try:
        img = Image.open(img_path).convert('RGB')
        img = img.resize((224, 224))  # VGG16 input size
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        feat = vgg.predict(x)
        return feat.flatten()
    except:
        # Return zeros if image can't be read
        return np.zeros((512,))

# Get list of image paths
image_paths = [os.path.join(image_folder, fname) for fname in os.listdir(image_folder) if fname.lower().endswith(('png','jpg','jpeg'))]

# Extract embeddings
image_features = []
for path in tqdm(image_paths):
    image_features.append(get_image_embedding_from_path(path))

# Convert to numpy array if needed
image_features = np.array(image_features)
print("Shape of image features:", image_features.shape)


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


  0%|          | 0/1000 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


  0%|          | 1/1000 [00:04<1:08:21,  4.11s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


100%|█████████▉| 999/1000 [02:23<00:00, 10.22it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


100%|██████████| 1000/1000 [02:23<00:00,  6.95it/s]

Shape of image features: (1000, 512)


In [ ]:
text_features[0].shape

(768,)

In [ ]:
X_text = np.array(text_features)
X_img = np.array(image_features)

X = np.concatenate([X_text, X_img], axis=1)  # shape: (n_samples, 384+512=896)
y = df_train['price'].values

In [14]:
X = np.array(text_features)
# X_img = np.array(image_features)

# X = np.concatenate([X_text, X_img], axis=1)  # shape: (n_samples, 384+512=896)
y = df_train['price'].values

In [11]:
def smape(y_true, y_pred):
    """
    Symmetric Mean Absolute Percentage Error
    Works as a custom loss function for regression.
    """
    epsilon = 1e-6  # to avoid division by zero
    numerator = tf.abs(y_true - y_pred)
    denominator = (tf.abs(y_true) + tf.abs(y_pred) + epsilon) / 2.0
    smape_value = 100 * tf.reduce_mean(numerator / denominator)
    return smape_value

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization,Input

model = Sequential([
    Input(shape=(X.shape[1],)),
    Dense(2000, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(1024, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(752, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5) ,
    Dense(256, activation='relu'),
    BatchNormalization(),

    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1,activation='linear')   # output: predicted price
])

model.compile(optimizer='adam', loss='mse',metrics=['mae',smape])
model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 2000)           │       770,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 2000)           │         8,000 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1024)           │     2,049,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 752)            │       770,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 752)            │         3,008 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 752)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 512)            │       385,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,166,081 (15.89 MB)

 Trainable params: 4,156,993 (15.86 MB)

 Non-trainable params: 9,088 (35.50 KB)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization,Input

model = Sequential([
    Input(shape=(X.shape[1],)),
    Dense(2000, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(1024, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(752, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5) ,
    Dense(256, activation='relu'),
    BatchNormalization(),

    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1,activation='linear')   # output: predicted price
])

model.compile(optimizer='adam', loss=smape,metrics=['mae', 'mse'])
model.summary()



Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_50 (Dense)                │ (None, 2000)           │     4,098,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_34          │ (None, 2000)           │         8,000 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25 (Dropout)            │ (None, 2000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_51 (Dense)                │ (None, 1024)           │     2,049,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_35          │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_52 (Dense)                │ (None, 752)            │       770,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_36          │ (None, 752)            │         3,008 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 752)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_53 (Dense)                │ (None, 512)            │       385,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_37          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_28 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_54 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_38          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_55 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_56 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_57 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,494,081 (28.59 MB)

 Trainable params: 7,484,993 (28.55 MB)

 Non-trainable params: 9,088 (35.50 KB)

In [15]:
model.fit(X, y, epochs=50, batch_size=16, validation_split=0.2)

Epoch 1/50
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 27s 5ms/step - loss: 1115.7596 - mae: 17.3552 - smape: 81.9893 - val_loss: 865.4344 - val_mae: 15.6313 - val_smape: 78.4063
Epoch 2/50
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 917.6603 - mae: 16.1940 - smape: 80.4840 - val_loss: 1324.0302 - val_mae: 17.5149 - val_smape: 81.1144
Epoch 3/50
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 873.2487 - mae: 15.8647 - smape: 80.3312 - val_loss: 1284.6367 - val_mae: 16.6061 - val_smape: 81.5739
Epoch 4/50
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - loss: 867.2725 - mae: 15.6441 - smape: 80.3674 - val_loss: 2245.9736 - val_mae: 17.1185 - val_smape: 78.0158
Epoch 5/50
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 762.8307 - mae: 15.2989 - smape: 80.0540 - val_loss: 1438.5797 - val_mae: 16.6065 - val_smape: 80.2624
Epoch 6/50
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 864.4496 - mae: 15.2883 - smape: 80.0858 - val_loss: 1524.3580 - val_mae: 17.2864 - val_smape: 80.9090
Epoc

In [ ]:
df_test_X=pd.df_train=pd.read_csv(url,nrows=20000)

In [ ]:
text_x=df_test_X.loc[10000:11000,:]

In [ ]:
text_x

,sample_id,catalog_content,image_link,price
10000,144360,Item Name: Oak Smoke Decaf Black Tea (Loose) (...,https://m.media-amazon.com/images/I/81GdsREBZw...,27.40
10001,31120,Item Name: International Collection Virgin Fla...,https://m.media-amazon.com/images/I/813xJZ-8Z5...,12.49
10002,239791,Item Name: Warheads Limited Edition Halloween ...,https://m.media-amazon.com/images/I/81zerWnK-A...,12.99
10003,90802,"Item Name: Swad Cardamom Seed, 14 Ounce\nBulle...",https://m.media-amazon.com/images/I/91UA80t+nK...,25.99
10004,114112,"Item Name: Ruffles Potato Chips, Lime & Jalape...",https://m.media-amazon.com/images/I/71l0W92MCB...,4.79
...,...,...,...,...
10996,17609,Item Name: Eight O'Clock Coffee Donut Shop Gro...,https://m.media-amazon.com/images/I/81ThYL4cvG...,14.32
10997,144717,"Item Name: La Croix Sparkling Water, Passion F...",https://m.media-amazon.com/images/I/81jzN2pYxz...,4.19
10998,37050,Item Name: Progresso Garden Vegetable Classics...,https://m.media-amazon.com/images/I/917-Tz1ROZ...,2.38
10999,115527,Item Name: San Antonio Dulce de Leche Classic ...,https://m.media-amazon.com/images/I/71U0zbHKUv...,15.99


In [ ]:
textinp=get_text_embeddings_mpnet(text_x['catalog_content'].tolist())

100%|██████████| 1001/1001 [00:22<00:00, 44.81it/s]


In [ ]:
testimage = [get_image_embedding_effnet_imageLink(url) for url in tqdm(text_x['image_link'])]
testimage= np.array(testimage)

100%|██████████| 1001/1001 [03:01<00:00,  5.50it/s]


In [ ]:
def smape(y_true, y_pred):
    return 100 * np.mean(np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2))

In [ ]:
ytext1=text_x['price']

In [ ]:
preds_df = pd.DataFrame(preds, columns=['pred'])

# Concatenate along columns
mdfnew = pd.concat([preds_df, ytext1.reset_index(drop=True)], axis=1)

In [ ]:
mdfnew

,pred,price
0,2.991906,3.920
1,4.596334,10.465
2,2.725857,3.420
3,29.862772,39.990
4,39.728313,85.420
...,...,...
995,28.752169,22.990
996,20.369280,4.290
997,8.005280,9.900
998,23.428749,4.960


In [ ]:
dfnew

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
0,2.991906,4.596334,2.725857,29.862772,39.728313,23.990362,20.690731,29.795309,23.858717,10.128128,...,22.460979,44.091972,18.11034,7.570084,23.075397,28.752169,20.36928,8.00528,23.428749,16.415594
1,3.920000,10.465000,3.420000,39.990000,85.420000,7.710000,1.480000,19.980000,35.990000,33.300000,...,8.745000,46.890000,5.49000,5.870000,23.500000,22.990000,4.29000,9.90000,4.960000,12.990000


In [ ]:
print("Valid", smape(y, ytext1))

Validation SMAPE: 94.53380319391648


IF Want to train with eff net bhi train ho toh ye

Ahh ab samajh gaya Keshav 😎 — tu basically **multi-modal regression** karna chahta hai:

* Input 1 → **Image** (EfficientNet fine-tune ho raha ho)
* Input 2 → **Text embeddings** (precomputed, e.g., MPNet 768-dim)
* Output → **Price prediction**

Yes, ye bilkul **possible hai**. Bas hume **Keras Functional API** ka use karna padega, kyunki Sequential API me multiple inputs directly handle nahi hote.

Main tujhe **step-by-step structure** bata deta hoon:

---

## 1️⃣ Image Input → EfficientNet

```python
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D, Input, Concatenate
from tensorflow.keras.models import Model
import tensorflow as tf

# Image input
img_input = Input(shape=(224,224,3))

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=img_input)

# Freeze all layers first
for layer in base_model.layers:
    layer.trainable = False

# Fine-tune last 4 layers
for layer in base_model.layers[-4:]:
    layer.trainable = True

x_img = base_model.output
x_img = GlobalAveragePooling2D()(x_img)  # image embedding vector
x_img = Dense(512, activation='relu')(x_img)
x_img = BatchNormalization()(x_img)
x_img = Dropout(0.5)(x_img)
```

---

## 2️⃣ Text Input → Precomputed embeddings

```python
# Suppose text embeddings = 768-dim
text_input = Input(shape=(768,))
x_text = Dense(512, activation='relu')(text_input)
x_text = BatchNormalization()(x_text)
x_text = Dropout(0.5)(x_text)
```

---

## 3️⃣ Combine Image + Text

```python
combined = Concatenate()([x_img, x_text])
x = Dense(512, activation='relu')(combined)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='linear')(x)
```

---

## 4️⃣ Create Model & Compile

```python
model = Model(inputs=[img_input, text_input], outputs=output)

def smape(y_true, y_pred):
    epsilon = 1e-6
    numerator = tf.abs(y_true - y_pred)
    denominator = (tf.abs(y_true) + tf.abs(y_pred) + epsilon) / 2.0
    return 100 * tf.reduce_mean(numerator / denominator)

model.compile(optimizer='adam', loss=smape, metrics=['mae','mse'])
model.summary()
```

---

## 5️⃣ Training

```python
# Suppose X_img = images as numpy array (224,224,3)
# X_text = precomputed embeddings (768-dim)
# y = price

model.fit([X_img, X_text], y, batch_size=32, epochs=10)
```

---

### 🔹 Notes:

1. EfficientNet **last 4 layers trainable** → model images ke features update karega
2. Text embeddings **freeze** → fast aur memory efficient, agar chahe to text branch bhi trainable kar sakta hai
3. Concatenate → dono modalities ke features ek saath regression head me jaate hain
4. Agar GPU memory kam ho → fine-tune **last 2 layers** instead of 4

---

Agar tu chaahe to mai **tere existing image folder + text embeddings se full working multi-modal model ka ready-to-run code** bana du jisme EfficientNet fine-tune + text embeddings input ho, aur tu seedha train kar sake.

Chaahe mai wo bana du?


In [ ]:
n